# Can we fit the CFFs using DNNs as predicted by a model?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Libraries:

In [ ]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import iminuit

import gepard as g
from gepard.fits import th_KM15

from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

### (1.2): Printing Library Versions:

In [ ]:
print(f"gepard version: {g.__version__}")
print(f"Tensorflow version: {tf.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"imunuit version: {iminuit.__version__}")

### (1.3): Matplotlib Plotting Aesthetics:

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
})

plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True

## (2): Generating Pseudodata:

### (2.0): Data Collection Definitions:

In [ ]:
VERSION_NUMBER = 1
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

### (2.1): Defining the Kinematic Boundaries:

#### (2.1.1): Defining *specifically* the Kinematic Settings:

In [ ]:
# kinematic boundaries
BEAM_K_LOWER = 5.5
BEAM_K_UPPER = 11.0
Q_SQUARED_LOWER = 1.0
Q_SQUARED_UPPER = 5.0
X_B_LOWER = 0.1
X_B_UPPER = 0.6
T_LOWER = -1.0
T_UPPER = -.1

# number of points for each variable
NUMBER_OF_BEAM_K = 6
NUMBER_OF_Q_SQUARED = 10
NUMBER_OF_X_B = 6
NUMBER_OF_T = 6

# iterable ranges:
K_RANGE = np.linspace(BEAM_K_LOWER, BEAM_K_UPPER, NUMBER_OF_BEAM_K)
Q2_RANGE = np.linspace(Q_SQUARED_LOWER, Q_SQUARED_UPPER, NUMBER_OF_Q_SQUARED)
X_B_RANGE = np.linspace(X_B_LOWER, X_B_UPPER, NUMBER_OF_X_B)
T_RANGE = np.linspace(T_LOWER, T_UPPER, NUMBER_OF_T)

#### (2.1.2): Defining *specifically* the Phi Values:

In [ ]:
NUMBER_OF_PHI_POINTS = 15
STARTING_PHI_VALUE_IN_DEGREES = 0.
ENDING_PHI_VALUE_IN_DEGREES = 360.

phi_array_in_degrees = np.linspace(
    start = STARTING_PHI_VALUE_IN_DEGREES,
    stop = ENDING_PHI_VALUE_IN_DEGREES,
    num = NUMBER_OF_PHI_POINTS)

phi_array_in_radians = [np.radians(degree_value) for degree_value in phi_array_in_degrees]

print(f"We have constructed a Python list of length {len(phi_array_in_radians)} of azimuthal angles from {STARTING_PHI_VALUE_IN_DEGREES} degrees to {ENDING_PHI_VALUE_IN_DEGREES} degrees.")

In [ ]:
def generate_latex_table():

    data = [
        ["$k$", 5.5, 11.0, 6],
        ["$Q^2$", 1.0, 5.0, 10],
        ["$x_B$", 0.1, 0.6, 6],
        ["$t$", -1.0, -0.1, 6],
        ["$\\phi$", 0.0, 360.0, 15]
    ]

    latex = "\\begin{table}[h]\n\\centering\n"
    latex += "\\begin{tabular}{lccc}\n\\hline\n"
    latex += "Variable & Min Value & Max Value & Total Count \\\\ \\hline\n"
    
    for row in data:
        latex += f"{row[0]} & {row[1]} & {row[2]} & {row[3]} \\\\\n"
    
    latex += "\\hline\n\\end{tabular}\n"
    latex += "\\caption{Summary of Kinematic Variables}\n"
    latex += "\\end{table}"
    
    return latex

print(generate_latex_table())


### (2.2): Making the Function to Generate Pseudodata:
**(WARNING: Running this function requires major computing resources!)**

In [ ]:
rows = []
set_counter = 0 
total_kinematic_settings = NUMBER_OF_BEAM_K * NUMBER_OF_Q_SQUARED * NUMBER_OF_X_B * NUMBER_OF_T
total_settings = NUMBER_OF_BEAM_K * NUMBER_OF_Q_SQUARED * NUMBER_OF_X_B * NUMBER_OF_T * NUMBER_OF_PHI_POINTS

print(f"[INFO]: Ready to iterate over {total_kinematic_settings} kinematic settings.")
print(f"[INFO]: A total of {total_settings} combinations points.")

for fixed_k in K_RANGE:
    for fixed_q_squared in Q2_RANGE:
        for fixed_x_bjorken in X_B_RANGE:
            for fixed_t in T_RANGE:

                # increment set counter
                set_counter += 1

                current_kinematic_setting = [g.DataPoint(
                    xB = fixed_x_bjorken, t = fixed_t, Q2 = fixed_q_squared, phi = fixed_phi,
                    process = "ep2epgamma", exptype = 'fixed target',
                    in1energy = fixed_k, in1charge = -1, in1polarization = +1,
                    observable = 'XS',
                    fname = 'Trento') for fixed_phi in phi_array_in_radians]
                
                ##########################################
                # Predicting CFFs at datapoint:
                ##########################################
                # cff H
                real_h_values = [th_KM15.ReH(datapoint) for datapoint in current_kinematic_setting]
                imag_h_values = [th_KM15.ImH(datapoint) for datapoint in current_kinematic_setting]

                # cff E
                real_e_values = [th_KM15.ReE(datapoint) for datapoint in current_kinematic_setting]
                imag_e_values = [th_KM15.ImE(datapoint) for datapoint in current_kinematic_setting]

                # cff Ht
                real_ht_values = [th_KM15.ReHt(datapoint) for datapoint in current_kinematic_setting]
                imag_ht_values = [th_KM15.ImHt(datapoint) for datapoint in current_kinematic_setting]

                # cff Et
                real_et_values = [th_KM15.ReEt(datapoint) for datapoint in current_kinematic_setting]
                imag_et_values = [th_KM15.ImEt(datapoint) for datapoint in current_kinematic_setting]

                CFF_H_KM15 = complex(real_h_values[0], imag_h_values[0])
                CFF_H_TILDE_KM15 = complex(real_ht_values[0], imag_ht_values[0])
                CFF_E_KM15 = complex(real_e_values[0], imag_e_values[0])
                CFF_E_TILDE_KM15 = complex(real_et_values[0], imag_et_values[0])

                ##########################################
                # Obtaining Observable Values:
                ##########################################

                km15_cross_section = DifferentialCrossSection(
                    configuration = {
                        "kinematics": BKM10Inputs(
                            lab_kinematics_k = fixed_k,
                            squared_Q_momentum_transfer = fixed_q_squared,
                            x_Bjorken = fixed_x_bjorken,
                            squared_hadronic_momentum_transfer_t = fixed_t),
                        "cff_inputs": CFFInputs(
                            compton_form_factor_h = CFF_H_KM15,
                            compton_form_factor_h_tilde = CFF_H_TILDE_KM15,
                            compton_form_factor_e = CFF_E_KM15,
                            compton_form_factor_e_tilde = CFF_E_TILDE_KM15),
                        "using_ww": True
                    },
                    verbose = False,
                    debugging = False)
                    
                ##########################################
                # d^{4}\sigma(lambda = 0, Lambda = 0)
                ##########################################

                bkm10_unp_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = 0.0,
                    target_polarization = 0.0).real
        
                ##########################################
                # d^{4}\sigma(lambda = +1, Lambda = 0)
                ##########################################
                    
                bkm10_plus_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = +1.0,
                    target_polarization = 0.0).real
                    
                ##########################################
                # d^{4}\sigma(lambda = -1, Lambda = 0)
                ##########################################

                bkm10_minus_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = -1.0,
                    target_polarization = 0.0).real
                    
                ##########################################
                # d^{4}\sigma(lambda = 0, Lambda = 1/2)
                ##########################################

                bkm10_unp_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = 0.0,
                    target_polarization = +0.5).real
                    
                ##########################################
                # d^{4}\sigma(lambda = +1, Lambda = 1/2)
                ##########################################
                    
                bkm10_plus_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = +1.0,
                    target_polarization = +0.5).real
                    
                ##########################################
                # d^{4}\sigma(lambda = -1, Lambda = 1/2)
                ##########################################

                bkm10_minus_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
                    phi_array_in_radians,
                    lepton_helicity = -1.0,
                    target_polarization = +0.5).real

                ##########################################
                # BSA(Lambda = 0)
                ##########################################
                    
                bkm10_bsa_km15 = km15_cross_section.compute_bsa(
                    phi_array_in_radians, 
                    target_polarization = 0.0).real

                ##########################################
                # BSA(Lambda = 1/2)
                ##########################################
                    
                bkm10_plus_lp_bsa_km15 = km15_cross_section.compute_bsa(
                    phi_array_in_radians,
                    target_polarization = +0.5).real

                ##########################################
                # BSA(Lambda = -1/2)
                ##########################################
                    
                bkm10_minus_lp_bsa_km15 = km15_cross_section.compute_bsa(
                    phi_array_in_radians,
                    target_polarization = -0.5).real

                # ##########################################
                # # TSA(lambda = 0.0)
                # ##########################################

                bkm10_tsa_km15 = km15_cross_section.compute_tsa(
                    phi_array_in_radians,
                    lepton_polarization = 0.0).real

                # ##########################################
                # # TSA(lambda = +1.0)
                # ##########################################

                bkm10_plus_beam_tsa_km15 = km15_cross_section.compute_tsa(
                    phi_array_in_radians,
                    lepton_polarization = +1.0).real

                # ##########################################
                # # TSA(lambda = -1.0)
                # ##########################################

                bkm10_minus_beam_tsa_km15 = km15_cross_section.compute_tsa(
                    phi_array_in_radians,
                    lepton_polarization = -1.0).real

                # ##########################################
                # # DSA
                # ##########################################

                bkm10_dsa_km15 = km15_cross_section.compute_dsa(
                    phi_array_in_radians).real

                if not np.all(np.isfinite(bkm10_unp_beam_unp_target_km15)):
                    print("[ERROR]: KM15 observables are unphysical: not finite")
                    continue

                if np.any((bkm10_unp_beam_unp_target_km15 < 0.)):
                    print("[ERROR]: KM15 observables are unphysical: cross-sections are negative")
                    continue
                    
                if np.any(np.isnan(bkm10_unp_beam_unp_target_km15)):
                    print("[ERROR]: KM15 observables are unphysical: encountered NaN")
                    continue
                
                for phi_index, phi_value in enumerate(phi_array_in_radians):
                    rows.append({
                        "set": set_counter,
                        "k": fixed_k,
                        "q_squared": fixed_q_squared,
                        "x_b": fixed_x_bjorken,
                        "t": fixed_t,
                        "phi": phi_value,
                        ####### IMPORTANT! Using KM15 VALUES! #######
                        "unp_beam_unp_target_xsec": bkm10_unp_beam_unp_target_km15[phi_index],
                        "plus_beam_unp_target_xsec": bkm10_plus_beam_unp_target_km15[phi_index],
                        "plus_minus_unp_target_xsec": bkm10_minus_beam_unp_target_km15[phi_index],
                        "unp_beam_unp_lp_xsec": bkm10_unp_beam_lp_target_km15[phi_index],
                        "plus_beam_unp_lp_xsec": bkm10_plus_beam_lp_target_km15[phi_index],
                        "plus_minus_unp_lp_xsec": bkm10_minus_beam_lp_target_km15[phi_index],
                        "unp_target_bsa": bkm10_bsa_km15[phi_index],
                        ####### IMPORTANT! Using KM15 VALUES! #######
                        "Re[H]": real_h_values[0], # KM15 value for Re[H]
                        "Im[H]": imag_h_values[0], # KM15 value for Im[H]
                        "Re[E]": real_e_values[0],
                        "Im[E]": imag_e_values[0],
                        "Re[Ht]": real_ht_values[0],
                        "Im[Ht]": imag_ht_values[0],
                        "Re[Et]": real_et_values[0],
                        "Im[Et]": imag_et_values[0]
                    })
                
                # clean up:
                del km15_cross_section
                del current_kinematic_setting
                
print(f"Has iteration successfully executed: {set_counter == total_kinematic_settings}")

print(len(rows))
kinematic_grid_data = pd.DataFrame(rows)
kinematic_grid_data.to_csv("do_not_open_kinematic_scan_v1.csv")

#### (2.2.1): Printing out some Data About the Datafile:

In [ ]:
print(f"[INFO]: Saved {len(kinematic_grid_data)} rows.")
print(f"[INFO]: Attempted {total_settings} total kinematic settings.")
print(f"[INFO]: Angles per setting: {NUMBER_OF_PHI_POINTS}")

### (2.3): Plots of the Input Space

#### (2.3.1): Extracting the Columns with the Kinematics:

In [ ]:
unique_kinematic_sets_dataframe = kinematic_grid_data.groupby(['set'], as_index = False).mean()
number_of_unique_kinematic_sets = len(unique_kinematic_sets_dataframe)
print(f"[INFO]: Total number of unique sets: {number_of_unique_kinematic_sets}")
# if the assert below is False, it means some settings were unphysical --- that's all!
print(f"[ASSERT]: Does it match the expected number? {number_of_unique_kinematic_sets == total_kinematic_settings}")

q_squared_column = unique_kinematic_sets_dataframe["q_squared"]
xb_column = unique_kinematic_sets_dataframe['x_b']
t_column = unique_kinematic_sets_dataframe["t"]

#### (2.3.2): 3D Scatterplot of $x_{\text{B}}$, $-t$, and $Q^{2}$

In [ ]:
input_space_figure = plt.figure(figsize = (9, 7))
input_space_axis = input_space_figure.add_subplot(1, 1, 1, projection = "3d")
input_space_axis.scatter(xb_column, q_squared_column, t_column, alpha = 0.3, s = 6.4)
input_space_axis.set_xlabel(r"$x_{\textrm{B}}$")
input_space_axis.set_ylabel(r"$Q^{2}$")
input_space_axis.set_zlabel(r"$-t$")
input_space_axis.set_title("Kinematic Settings Space")

input_space_figure.savefig(f"version_{VERSION_NUMBER}/plots/kinematic_settings_space_v{MAJOR_MINOR_NUMBER}.png")
input_space_figure.savefig(f"version_{VERSION_NUMBER}/plots/kinematic_settings_space_v{MAJOR_MINOR_NUMBER}.eps")

#### (2.3.3): 2D Scatterplot of $-t$ vs. $x_{\text{B}}$ Domain

In [ ]:
t_vs_xb_figure, t_vs_x_axis = plt.subplots(1, 1, figsize = (8, 7))
t_vs_x_axis.scatter(xb_column, t_column, alpha = 0.3, s = 10.4)
t_vs_x_axis.set_xlabel(r"$x_{\textrm{B}}$")
t_vs_x_axis.set_ylabel(r"$-t$")
t_vs_x_axis.set_title(r"$-t$ vs. $x_{\textrm{B}}$")

t_vs_xb_figure.savefig(f"version_{VERSION_NUMBER}/plots/t_vs_xb_v{MAJOR_MINOR_NUMBER}.png")
t_vs_xb_figure.savefig(f"version_{VERSION_NUMBER}/plots/t_vs_xb_v{MAJOR_MINOR_NUMBER}.eps")

#### (2.3.4): 2D Scatterplot of $Q^{2}$ vs. $-t$ Domain

In [ ]:
qsq_vs_t_figure, qsq_vs_t_axis = plt.subplots(1, 1, figsize = (8, 7))
qsq_vs_t_axis.scatter(q_squared_column, t_column, alpha = 0.1, s = 10.4)
qsq_vs_t_axis.set_xlabel(r"$Q^{2}$")
qsq_vs_t_axis.set_ylabel(r"$-t$")
qsq_vs_t_axis.set_title(r"$-t$ vs. $Q^{2}$")

qsq_vs_t_figure.savefig(f"version_{VERSION_NUMBER}/plots/qsquared_vs_t_v{MAJOR_MINOR_NUMBER}.png")
qsq_vs_t_figure.savefig(f"version_{VERSION_NUMBER}/plots/qsquared_vs_t_v{MAJOR_MINOR_NUMBER}.eps")

#### (2.3.5): 2D Scatterplot of $x_{\text{B}}$ vs. $Q^{2}$ Domain

In [ ]:
xb_vs_qsq_figure, xb_vs_qsq_axis = plt.subplots(1, 1, figsize = (8, 7))
xb_vs_qsq_axis.scatter(xb_column, q_squared_column, alpha = 0.1, s = 10.4)
xb_vs_qsq_axis.set_xlabel(r"$x_{\textrm{B}}$")
xb_vs_qsq_axis.set_ylabel(r"$Q^{2}$")
xb_vs_qsq_axis.set_title(r"$Q^{2}$ vs. $x_{\textrm{B}}$")

xb_vs_qsq_figure.savefig(f"version_{VERSION_NUMBER}/plots/xb_vs_qsquared_v{MAJOR_MINOR_NUMBER}.png")
xb_vs_qsq_figure.savefig(f"version_{VERSION_NUMBER}/plots/xb_vs_qsquared_v{MAJOR_MINOR_NUMBER}.eps")

#### (2.3.6): CFF $\mathcal{H}$ Trends vs. $x_{\text{B}}$, $Q^{2}$, and $-t$

##### (2.3.6.1): Extracting the Unique Kinematic Sets and some Data Analysis
**[NOTE]: This block takes a bit of time!**

In [ ]:
# smooth linspaces for KM15 interpolation:
smooth_xb_range = np.linspace(unique_kinematic_sets_dataframe["x_b"].min(), unique_kinematic_sets_dataframe["x_b"].max(), 100)
smooth_q_squared_range = np.linspace(unique_kinematic_sets_dataframe["q_squared"].min(), unique_kinematic_sets_dataframe["q_squared"].max(), 100)
smooth_t_range = np.linspace(unique_kinematic_sets_dataframe["t"].min(), unique_kinematic_sets_dataframe["t"].max(), 100)

# find the mean values of the kinematic settings:
dataframe_mean_t = unique_kinematic_sets_dataframe["t"].mean()
dataframe_stddev_t = unique_kinematic_sets_dataframe["t"].std()
dataframe_mean_xb = unique_kinematic_sets_dataframe["x_b"].mean()
dataframe_stddev_xb = unique_kinematic_sets_dataframe["x_b"].std()
dataframe_mean_qsq = unique_kinematic_sets_dataframe["q_squared"].mean()
dataframe_stddev_qsq = unique_kinematic_sets_dataframe["q_squared"].std()

print(f"[INFO]: Determined mean value of -t in generated dataframe: <-t> = {dataframe_mean_t} with stddev = {dataframe_stddev_t}")
print(f"[INFO]: Determined mean value of xb in generated dataframe: <xb> = {dataframe_mean_xb} with stddev = {dataframe_stddev_xb}")
print(f"[INFO]: Determined mean value of Q^2 in generated dataframe: <Q^2> = {dataframe_mean_qsq} with stddev = {dataframe_stddev_qsq}")

real_h_constant_qsq_t = [th_KM15.ReH(g.DataPoint(
    xB = xb_value, t = dataframe_mean_t, Q2 = dataframe_mean_qsq,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for xb_value in smooth_xb_range]

real_h_constant_xb_t = [th_KM15.ReH(g.DataPoint(
    xB = dataframe_mean_xb, t = dataframe_mean_t, Q2 = qsq_value,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for qsq_value in smooth_q_squared_range]

real_h_constant_qsq_xb = [th_KM15.ReH(g.DataPoint(
    xB = dataframe_mean_xb, t = t_value, Q2 = dataframe_mean_qsq,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for t_value in smooth_t_range]

imag_h_constant_qsq_t = [th_KM15.ImH(g.DataPoint(
    xB = xb_value, t = dataframe_mean_t, Q2 = dataframe_mean_qsq,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for xb_value in smooth_xb_range]

imag_h_constant_xb_t = [th_KM15.ImH(g.DataPoint(
    xB = dataframe_mean_xb, t = dataframe_mean_t, Q2 = qsq_value,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for qsq_value in smooth_q_squared_range]

imag_h_constant_qsq_xb = [th_KM15.ImH(g.DataPoint(
    xB = dataframe_mean_xb, t = t_value, Q2 = dataframe_mean_qsq,
    process = "ep2epgamma", exptype = 'fixed target', in1energy = fixed_k, in1charge = -1, in1polarization = +1, observable = 'XS', fname = 'Trento')) 
    for t_value in smooth_t_range]

squared_mahalanobis_for_qsq = (
    ((unique_kinematic_sets_dataframe["t"] - dataframe_mean_t) / dataframe_stddev_t)**2 + 
    ((unique_kinematic_sets_dataframe["x_b"] - dataframe_mean_xb) / dataframe_stddev_xb)**2
)

squared_mahalanobis_for_t = (
    ((unique_kinematic_sets_dataframe["q_squared"] - dataframe_mean_qsq) / dataframe_stddev_qsq)**2 + 
    ((unique_kinematic_sets_dataframe["x_b"] - dataframe_mean_xb) / dataframe_stddev_xb)**2
)

squared_mahalanobis_for_xb = (
    ((unique_kinematic_sets_dataframe["t"] - dataframe_mean_t) / dataframe_stddev_t)**2 + 
    ((unique_kinematic_sets_dataframe["q_squared"] - dataframe_mean_qsq) / dataframe_stddev_qsq)**2
)

print(f"[INFO]: Total Mahalanobis Distance for Q^2 is: {squared_mahalanobis_for_qsq}")
print(f"[INFO]: Total Mahalanobis Distance for t is: {squared_mahalanobis_for_t}")
print(f"[INFO]: Total Mahalanobis Distance for xB is: {squared_mahalanobis_for_xb}")

closest_kinematic_setting_for_qsq = unique_kinematic_sets_dataframe.loc[
    squared_mahalanobis_for_qsq.idxmin(), # row
    'set' # column
    ]

closest_kinematic_setting_for_t = unique_kinematic_sets_dataframe.loc[
    squared_mahalanobis_for_t.idxmin(), # row
    'set' # column
    ]

closest_kinematic_setting_for_xb = unique_kinematic_sets_dataframe.loc[
    squared_mahalanobis_for_xb.idxmin(), # row
    'set' # column
    ]

print(f"[INFO]: The kinematic set number that corresponds with the smallest Mahalanobis Distance for Q^2 is: {closest_kinematic_setting_for_qsq}")
print(f"[INFO]: The kinematic set number that corresponds with the smallest Mahalanobis Distance for t is: {closest_kinematic_setting_for_t}")
print(f"[INFO]: The kinematic set number that corresponds with the smallest Mahalanobis Distance for xB is: {closest_kinematic_setting_for_xb}")

##### (2.3.6.2): The Actual Re[$\mathcal{H}$] Plots

In [ ]:
cff_h_figure, cff_h_axis = plt.subplots(2, 3, figsize = (12, 7))

cff_h_axis[0][0].scatter(unique_kinematic_sets_dataframe["x_b"], unique_kinematic_sets_dataframe["Re[H]"])
cff_h_axis[0][0].plot(smooth_xb_range, real_h_constant_qsq_t, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[0][0].set_xlabel(r"$x_{\textrm{B}}$")
cff_h_axis[0][0].set_ylabel(r"Re$[ \mathcal{H} ](x_{\textrm{B}}, \langle Q^{2} \rangle, \langle -t \rangle)$")
cff_h_axis[0][0].grid(True, alpha = 0.4)
cff_h_axis[0][0].legend(fontsize = 14)

cff_h_axis[0][1].scatter(unique_kinematic_sets_dataframe["q_squared"], unique_kinematic_sets_dataframe["Re[H]"])
cff_h_axis[0][1].plot(smooth_q_squared_range, real_h_constant_xb_t, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[0][1].set_xlabel(r"$Q^{2}$")
cff_h_axis[0][1].set_ylabel(r"Re$[ \mathcal{H} ](\langle x_{\textrm{B}} \rangle, Q^{2}, \langle -t \rangle)$")
cff_h_axis[0][1].grid(True, alpha = 0.4)
cff_h_axis[0][1].legend(fontsize = 14)

cff_h_axis[0][2].scatter(unique_kinematic_sets_dataframe["t"], unique_kinematic_sets_dataframe["Re[H]"])
cff_h_axis[0][2].plot(smooth_t_range, real_h_constant_qsq_xb, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[0][2].set_xlabel(r"$-t$")
cff_h_axis[0][2].set_ylabel(r"Re$[ \mathcal{H} ] (\langle x_{\textrm{B}} \rangle, \langle Q^{2} \rangle, -t)$")
cff_h_axis[0][2].grid(True, alpha = 0.4)
cff_h_axis[0][2].legend(fontsize = 14)

cff_h_axis[1][0].scatter(unique_kinematic_sets_dataframe["x_b"], unique_kinematic_sets_dataframe["Im[H]"])
cff_h_axis[1][0].plot(smooth_xb_range, imag_h_constant_qsq_t, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[1][0].set_xlabel(r"$x_{\textrm{B}}$")
cff_h_axis[1][0].set_ylabel(r"Im$[ \mathcal{H} ](x_{\textrm{B}}, \langle Q^{2} \rangle, \langle -t \rangle)$")
cff_h_axis[1][0].grid(True, alpha = 0.4)
cff_h_axis[1][0].legend(fontsize = 14)

cff_h_axis[1][1].scatter(unique_kinematic_sets_dataframe["q_squared"], unique_kinematic_sets_dataframe["Im[H]"])
cff_h_axis[1][1].plot(smooth_q_squared_range, imag_h_constant_xb_t, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[1][1].set_xlabel(r"$Q^{2}$")
cff_h_axis[1][1].set_ylabel(r"Im$[ \mathcal{H} ](\langle x_{\textrm{B}} \rangle, Q^{2}, \langle -t \rangle)$")
cff_h_axis[1][1].grid(True, alpha = 0.4)
cff_h_axis[1][1].legend(fontsize = 14)

cff_h_axis[1][2].scatter(unique_kinematic_sets_dataframe["t"], unique_kinematic_sets_dataframe["Im[H]"])
cff_h_axis[1][2].plot(smooth_t_range, imag_h_constant_qsq_xb, color = "red", linestyle = "--", label = "KM15")
cff_h_axis[1][2].set_xlabel(r"$-t$")
cff_h_axis[1][2].set_ylabel(r"Im$[ \mathcal{H} ] (\langle x_{\textrm{B}} \rangle, \langle Q^{2} \rangle, -t)$")
cff_h_axis[1][2].grid(True, alpha = 0.4)
cff_h_axis[1][2].legend(fontsize = 14)

# label the top y-axis:
# label the bottom y-axis:
# give the total plot a title:
cff_h_figure.suptitle(r'Trends of CFF $\mathcal{H}$ across Individual Kinematics', fontsize = 16)